# Heavy Pretrained Experiments

Notebook experimental non from-scratch pour viser un meilleur score Kaggle si le pretrained est autorise.

Modeles testes avec torchvision pretrained:
- ConvNeXt-Tiny
- EfficientNetV2-S
- Swin-Tiny

Chaque modele fait 5-fold, TTA, sauvegarde des probabilites par fold, CSV final, puis ensemble soft-probs.


## 1. Setup


In [ ]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR       = ROOT / 'data'
OUTPUT_DIR     = ROOT / 'outputs'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
SUBMISSION_DIR = OUTPUT_DIR / 'submissions'

for p in [CHECKPOINT_DIR, SUBMISSION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Root  :', ROOT)
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('ATTENTION: CUDA indisponible.')


## 2. Config

Pour tester rapidement, mets `EPOCHS = 2` et `N_SPLITS = 2`. Pour le vrai run, garde 5 folds.


In [ ]:
N_SPLITS = 5
NUM_CLASSES = 8
NUM_WORKERS = 0

IMG_SIZE = (384, 576)      # H, W, ratio textile preserve
RESIZE_SIZE = (432, 648)
BATCH_SIZE = 16            # safe pour A6000 avec Swin/EfficientNet; augmenter a 24/32 si VRAM OK

FREEZE_EPOCHS = 3          # warmup classifier head
EPOCHS = 35                # fine-tune complet; monter a 50 si le val monte encore
PATIENCE = 10

LR_HEAD = 2e-3
LR_BACKBONE = 2e-4
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.05

# Lancer dans cet ordre. ConvNeXt est prioritaire.
RUN_MODELS = [
    'convnext_tiny',
    'efficientnet_v2_s',
    'swin_t',
]

# Pour reprendre apres interruption: {'convnext_tiny': 3} reprend ConvNeXt au fold 3.
START_FOLD = {
    'convnext_tiny': 1,
    'efficientnet_v2_s': 1,
    'swin_t': 1,
}

print('Config OK')


## 3. Data


In [ ]:
def read_train_csv(path):
    try:
        df = pd.read_csv(path, sep=';')
        if len(df.columns) == 1:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df['id'] = df['id'].astype(int)
    df['label'] = df['label'].astype(int)
    return df


def resolve_image_path(folder, image_id):
    stem = str(int(image_id))
    for ext in ['.tif', '.tiff', '.png', '.jpg', '.jpeg']:
        p = folder / f'{stem}{ext}'
        if p.exists():
            return p
    matches = sorted(folder.glob(f'{stem}.*'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Image introuvable: {image_id}')

train_dir = DATA_DIR / 'train'
test_dir = DATA_DIR / 'test'
df = read_train_csv(DATA_DIR / 'train.csv')
df['path'] = df['id'].map(lambda x: resolve_image_path(train_dir, x))

test_paths = sorted(test_dir.glob('*.*'), key=lambda p: int(p.stem))
test_df = pd.DataFrame({'id': [int(p.stem) for p in test_paths], 'path': test_paths})

print(df.head())
print(df['label'].value_counts().sort_index())
print('Train images:', len(df))
print('Test images :', len(test_df))
print('Example:', Image.open(df.iloc[0]['path']).mode, Image.open(df.iloc[0]['path']).size)


## 4. Transforms et Dataset

On convertit le grayscale en 3 canaux pour utiliser les poids ImageNet.


In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(RESIZE_SIZE),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    transforms.RandomErasing(p=0.08, scale=(0.01, 0.04), ratio=(0.3, 3.3)),
])

eval_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TildaDataset(Dataset):
    def __init__(self, frame, transform=None, has_labels=True):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('L')
        if self.transform is not None:
            image = self.transform(image)
        label = int(row['label']) if self.has_labels else -1
        return image, label, int(row['id'])


def make_loader(frame, transform, batch_size=BATCH_SIZE, shuffle=False, has_labels=True):
    ds = TildaDataset(frame, transform=transform, has_labels=has_labels)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=torch.cuda.is_available())


## 5. Model factory


In [ ]:
def build_model(name, num_classes=8):
    if name == 'convnext_tiny':
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT
        model = models.convnext_tiny(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model

    if name == 'efficientnet_v2_s':
        weights = models.EfficientNet_V2_S_Weights.DEFAULT
        model = models.efficientnet_v2_s(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model

    if name == 'swin_t':
        weights = models.Swin_T_Weights.DEFAULT
        model = models.swin_t(weights=weights)
        in_features = model.head.in_features
        model.head = nn.Linear(in_features, num_classes)
        return model

    raise ValueError(name)


def set_backbone_trainable(model, trainable):
    for p in model.parameters():
        p.requires_grad = trainable


def set_head_trainable(model, name):
    if name in ['convnext_tiny', 'efficientnet_v2_s']:
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif name == 'swin_t':
        for p in model.head.parameters():
            p.requires_grad = True

for model_name in RUN_MODELS:
    m = build_model(model_name, NUM_CLASSES)
    print(model_name, f'{sum(p.numel() for p in m.parameters())/1e6:.1f}M params')
    del m


## 6. Train / predict


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, correct, n = 0.0, 0, 0
    for images, labels, _ in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        n += labels.size(0)
    return correct / n, total_loss / n


def fit_model(model, name, fold_name, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    best_acc = -1.0
    best_epoch = 0
    history = []
    best_path = CHECKPOINT_DIR / f'best_{fold_name}.pt'
    start = time.time()

    # Phase 1: head only
    set_backbone_trainable(model, False)
    set_head_trainable(model, name)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD,
                                  weight_decay=WEIGHT_DECAY)
    for ep in range(1, FREEZE_EPOCHS + 1):
        tr_acc, tr_loss = run_epoch(model, train_loader, criterion, optimizer)
        va_acc, va_loss = run_epoch(model, val_loader, criterion, None)
        history.append({'phase':'head', 'epoch':ep, 'train_acc':tr_acc, 'train_loss':tr_loss,
                        'val_acc':va_acc, 'val_loss':va_loss})
        if va_acc > best_acc:
            best_acc, best_epoch = va_acc, ep
            torch.save({'model_state_dict': model.state_dict(), 'best_acc': best_acc}, best_path)
        print(f'{fold_name} | head ep {ep:02d} | train {tr_acc:.4f}/{tr_loss:.4f} | val {va_acc:.4f}/{va_loss:.4f} | best {best_acc:.4f}')

    # Phase 2: full fine-tune
    set_backbone_trainable(model, True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_BACKBONE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_BACKBONE * 0.03)

    for ep in range(1, EPOCHS + 1):
        tr_acc, tr_loss = run_epoch(model, train_loader, criterion, optimizer)
        va_acc, va_loss = run_epoch(model, val_loader, criterion, None)
        scheduler.step()
        global_epoch = FREEZE_EPOCHS + ep
        history.append({'phase':'full', 'epoch':global_epoch, 'train_acc':tr_acc, 'train_loss':tr_loss,
                        'val_acc':va_acc, 'val_loss':va_loss, 'lr':scheduler.get_last_lr()[0]})
        if va_acc > best_acc:
            best_acc, best_epoch = va_acc, global_epoch
            torch.save({'model_state_dict': model.state_dict(), 'best_acc': best_acc}, best_path)
        print(f'{fold_name} | ft ep {ep:03d} | train {tr_acc:.4f}/{tr_loss:.4f} | val {va_acc:.4f}/{va_loss:.4f} | best {best_acc:.4f} @ {best_epoch}')
        if global_epoch - best_epoch >= PATIENCE:
            print(f'Early stop: no improvement for {PATIENCE} epochs')
            break

    return pd.DataFrame(history), best_path, best_acc, best_epoch, (time.time() - start) / 60


@torch.no_grad()
def predict_tta(model, loader):
    model.eval()
    all_probs, all_ids = [], []
    for images, _, image_ids in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        variants = [images, torch.flip(images, dims=[-1]), torch.flip(images, dims=[-2]), torch.flip(images, dims=[-2, -1])]
        probs = torch.stack([torch.softmax(model(v), dim=1) for v in variants]).mean(0)
        all_probs.append(probs.cpu())
        all_ids.extend(image_ids.numpy().tolist())
    return torch.cat(all_probs).numpy(), np.array(all_ids)


def save_submission(ids, probs, path):
    sub = pd.DataFrame({'id': ids, 'label': probs.argmax(1).astype(int)}).sort_values('id')
    sub.to_csv(path, index=False)
    print('Saved:', path)
    print(sub['label'].value_counts().sort_index())
    return sub


## 7. Run 5-fold par modele


In [ ]:
def run_model_5fold(model_name):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    test_loader = make_loader(test_df, eval_tfms, batch_size=BATCH_SIZE, shuffle=False, has_labels=False)
    fold_probs = []
    ids_reference = None
    rows = []
    start_fold = START_FOLD.get(model_name, 1)

    print('\n' + '=' * 80)
    print(f'RUN {model_name} from fold {start_fold}')
    print('=' * 80)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df['label']), start=1):
        fold_name = f'{model_name}_pretrained_fold{fold}'
        probs_path = CHECKPOINT_DIR / f'probs_{fold_name}.npy'
        ids_path = CHECKPOINT_DIR / f'ids_{fold_name}.npy'

        if fold < start_fold:
            if probs_path.exists() and ids_path.exists():
                probs = np.load(probs_path)
                ids = np.load(ids_path)
                fold_probs.append(probs)
                ids_reference = ids if ids_reference is None else ids_reference
                print(f'Fold {fold}: loaded {probs.shape}')
            else:
                print(f'Fold {fold}: skip requested but no npy found')
            continue

        tr_df = df.iloc[tr_idx].reset_index(drop=True)
        va_df = df.iloc[va_idx].reset_index(drop=True)
        train_loader = make_loader(tr_df, train_tfms, batch_size=BATCH_SIZE, shuffle=True, has_labels=True)
        val_loader = make_loader(va_df, eval_tfms, batch_size=BATCH_SIZE, shuffle=False, has_labels=True)

        model = build_model(model_name, NUM_CLASSES).to(DEVICE)
        hist, best_path, best_acc, best_epoch, elapsed = fit_model(model, model_name, fold_name, train_loader, val_loader)
        hist.to_csv(OUTPUT_DIR / f'history_{fold_name}.csv', index=False)

        ckpt = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        probs, ids = predict_tta(model, test_loader)
        np.save(probs_path, probs)
        np.save(ids_path, ids)

        if ids_reference is None:
            ids_reference = ids
        else:
            assert np.array_equal(ids_reference, ids)
        fold_probs.append(probs)

        save_submission(ids, probs, SUBMISSION_DIR / f'submission_{fold_name}_tta_labels0.csv')
        rows.append({'model': model_name, 'fold': fold, 'best_val_acc': best_acc,
                     'best_epoch': best_epoch, 'training_time_min': elapsed,
                     'checkpoint': str(best_path)})

        del model
        torch.cuda.empty_cache()

    result_df = pd.DataFrame(rows)
    result_df.to_csv(OUTPUT_DIR / f'results_{model_name}_pretrained.csv', index=False)
    display(result_df)
    if len(fold_probs) == N_SPLITS:
        mean_probs = np.mean(fold_probs, axis=0)
        np.save(CHECKPOINT_DIR / f'probs_{model_name}_pretrained_5fold.npy', mean_probs)
        np.save(CHECKPOINT_DIR / f'ids_{model_name}_pretrained_5fold.npy', ids_reference)
        save_submission(ids_reference, mean_probs, SUBMISSION_DIR / f'submission_{model_name}_pretrained_5fold_tta_labels0.csv')
    else:
        print(f'{model_name}: {len(fold_probs)}/{N_SPLITS} folds only')
    return result_df

all_results = []
for model_name in RUN_MODELS:
    all_results.append(run_model_5fold(model_name))

summary = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
summary.to_csv(OUTPUT_DIR / 'results_heavy_pretrained.csv', index=False)
display(summary)
if not summary.empty:
    display(summary.groupby('model')['best_val_acc'].agg(['count', 'mean', 'std', 'min', 'max']))


## 8. Ensemble des pretrained disponibles


In [ ]:
probs_list = []
ids_reference = None
used = []
for model_name in RUN_MODELS:
    probs_path = CHECKPOINT_DIR / f'probs_{model_name}_pretrained_5fold.npy'
    ids_path = CHECKPOINT_DIR / f'ids_{model_name}_pretrained_5fold.npy'
    if probs_path.exists() and ids_path.exists():
        probs = np.load(probs_path)
        ids = np.load(ids_path)
        if ids_reference is None:
            ids_reference = ids
        else:
            assert np.array_equal(ids_reference, ids)
        probs_list.append(probs)
        used.append(model_name)

if probs_list:
    ensemble_probs = np.mean(probs_list, axis=0)
    np.save(CHECKPOINT_DIR / 'probs_heavy_pretrained_ensemble.npy', ensemble_probs)
    save_submission(ids_reference, ensemble_probs,
                    SUBMISSION_DIR / 'submission_heavy_pretrained_ensemble_tta_labels0.csv')
    print('Used:', used)
else:
    print('No complete pretrained 5-fold probabilities found yet.')
